In [ ]:
import ee
import math
import time
from datetime import datetime
from io import BytesIO
import requests
from PIL import Image, ImageDraw
ee.Initialize()

In [ ]:
# --- Lat/lon <-> tile number conversion (standard Web Mercator formula) ---
def latlon_to_tile(lat, lon, zoom):
    """Convert latitude/longitude to tile coordinates at a given zoom level."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convert lat/lon to absolute PIXEL coordinates (not just tile) at the given zoom level.
    This allows locating the exact point within the canvas, not just the tile."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    # Calculate pixel coordinates in the global pixel space
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel


def draw_marker(canvas, x, y, radius=8, color=(220, 30, 30)):
    """Draw a circular marker with white outline at position (x, y) on the canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )


# --- Download and stitch tiles around a point ---
def get_point_map(lat, lon, zoom=17, radius=2, out_prefix="point_map", show_marker=True):
    """
    Generate a map centered on a specific point by downloading and stitching map tiles.
    
    Parameters:
        lat, lon: coordinates of the center point
        zoom: zoom level (OpenTopoMap supports up to 17)
        radius: how many tiles to add around the center in each direction.
                radius=2 -> 5x5 tile grid
        show_marker: if True, draws a red dot at the exact location
        out_prefix: base name for the output file; automatically appended with -vYYMMDDHHMMSS.png
    
    Returns:
        str: path to the saved image file
    """
    # Generate timestamp at the moment of map creation: YYMMDDHHMMSS (e.g., 260803143022)
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    out_path = f"{out_prefix}-v{timestamp}.png"

    # Calculate the tile coordinates for the center point
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    
    # Define the range of tiles to download (creating a grid around the center)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    # Calculate the total canvas size based on the tile grid
    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    # Set a user agent header to identify our application to the tile server
    headers = {"User-Agent": "agri_land_suitability_pipeline (your_email@example.com)"}

    # Download each tile in the grid
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            # Construct the URL for the tile from OpenTopoMap
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            # Handle failed tile downloads
            if resp.status_code != 200:
                print(f"Tile {x},{y} failed with status {resp.status_code}")
                time.sleep(0.5)
                continue

            # Try to open and paste the tile onto the canvas
            try:
                tile_img = Image.open(BytesIO(resp.content))
                # Calculate the position where this tile should be placed on the canvas
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} failed: {e}")

            # Respect the rate limit (~2 requests/second max)
            time.sleep(0.5)

    # Draw a marker at the exact point location if requested
    if show_marker:
        # Get the absolute pixel position of the point in the global pixel space
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        # Convert to canvas-relative coordinates by subtracting the canvas origin
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        # Draw the marker at the calculated position
        draw_marker(canvas, px_canvas, py_canvas)

    # Save the final composite image
    canvas.save(out_path)
    print(f"Map saved to {out_path} ({width}x{height}px)")
    return out_path


# Test the function with example coordinates
get_point_map(3.580109040361371, -76.31299479308868, zoom=17, radius=2)